In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage 
import meep as mp

# ==============================================================================
# SECTION 0: GLOBAL PARAMETERS (Updated based on your new script)
# ==============================================================================

RESOLUTION = 40
DPML = 1.0
MDM_LC = 4.0
WG_LENGTH = 0.5
MULTIMODE_WG_WIDTH = 1.0
SINGLEMODE_WG_WIDTH = 0.5

SX = 2 * DPML + MDM_LC + 2 * WG_LENGTH
SY = 2 * DPML + MDM_LC
CELL = mp.Vector3(SX, SY, 0)

output_port1_y = 1.0
output_port2_y = -1.0

N_SIO2 = 1.44
N_SI = 3.48
SIO2_MEDIUM = mp.Medium(index=N_SIO2)
SI_MEDIUM = mp.Medium(index=N_SI)

wl_cen = 1.55 
FCEN = 1 / wl_cen
DF = 0.1 * FCEN
NFREQ = 1

KERNEL_SIZE = 5
KERNEL_SIGMA = 1.0
TANH_BETA = 50
TANH_ETA = 0.5

# ==============================================================================
# SECTION 1: GEOMETRY & HELPERS
# ==============================================================================

def gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA):
    ax = np.arange(-(size//2), size//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= np.sum(kernel)
    return kernel

def tanh_projection(x, beta=TANH_BETA, eta=TANH_ETA):
    num = np.tanh(beta * eta) + np.tanh(beta * (x - eta))
    den = np.tanh(beta * eta) + np.tanh(beta * (1 - eta))
    return num / den

def get_projected_density_matrix(binary_vector, grid_rows, grid_cols):
    grid_matrix = np.array(binary_vector).astype(float).reshape((grid_rows, grid_cols))

    kernel = gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA)
    try:
        from scipy.signal import convolve2d
        density = convolve2d(grid_matrix, kernel, mode='same', boundary='symm')
    except Exception:
        k = kernel.shape[0]
        pad = k // 2
        img_p = np.pad(grid_matrix, ((pad, pad), (pad, pad)), mode='reflect')
        density = np.zeros_like(grid_matrix)
        for i in range(grid_rows):
            for j in range(grid_cols):
                density[i, j] = np.sum(img_p[i:i+k, j:j+k] * kernel)
    
    projected_density = tanh_projection(density, beta=TANH_BETA, eta=TANH_ETA)
    projected_density = np.clip(projected_density, 0.0, 1.0)
    return projected_density

def create_projected_geometry(binary_vector, grid_rows, grid_cols):
    density = get_projected_density_matrix(binary_vector, grid_rows, grid_cols)
    weights = density.flatten()
    material_grid = mp.MaterialGrid(mp.Vector3(grid_cols, grid_rows), SIO2_MEDIUM, SI_MEDIUM, weights=weights)
    material_grid.smoothing_radius = 1.0 
    design_block = mp.Block(size=mp.Vector3(MDM_LC, MDM_LC, mp.inf), center=mp.Vector3(), material=material_grid)
    return [design_block]

# ==============================================================================
# SECTION 2: SIMULATION & PLOTTING
# ==============================================================================

def replot_from_npy(npy_file, output_folder, grid_rows=16, grid_cols=16):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    print(f"Loading data from: {npy_file}")
    
    # 讀取 npy 檔案
    best_config = np.load(npy_file)
    
    # 自動推斷大小 (若檔案為 2D 陣列)
    if len(best_config.shape) == 2:
        grid_rows, grid_cols = best_config.shape
        best_config = best_config.flatten()
        print(f"Auto-detected grid size: {grid_rows} x {grid_cols}")
    else:
        print(f"Loaded 1D array. Using provided grid size: {grid_rows} x {grid_cols}")
    
    print(f"\n>>> Starting Detailed MEEP Analysis & Plotting <<<")
    
    mdm_structure = create_projected_geometry(best_config, grid_rows, grid_cols)
    
    input_wg_center_x = -MDM_LC / 2 - (WG_LENGTH + DPML) / 2
    output_wg_center_x = MDM_LC / 2 + (WG_LENGTH + DPML) / 2
    
    fixed_geometry = [
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML, MULTIMODE_WG_WIDTH, mp.inf), center=mp.Vector3(input_wg_center_x, 0), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML, SINGLEMODE_WG_WIDTH, mp.inf), center=mp.Vector3(output_wg_center_x, output_port1_y), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML, SINGLEMODE_WG_WIDTH, mp.inf), center=mp.Vector3(output_wg_center_x, output_port2_y), material=SI_MEDIUM),
    ]
    full_geometry = fixed_geometry + mdm_structure
    
    src_center = mp.Vector3(-SX / 2 + DPML + 0.01, 0)
    src_size = mp.Vector3(0, MULTIMODE_WG_WIDTH)
    monitor_size = mp.Vector3(0, 4 * SINGLEMODE_WG_WIDTH)
    monitor_x_pos = MDM_LC / 2 + WG_LENGTH / 3
    
    mode_definitions = {'TE0': {'band_num': 1, 'parity': mp.EVEN_Y}, 'TE1': {'band_num': 2, 'parity': mp.ODD_Y}}
    
    for mode_name, props in mode_definitions.items():
        print(f"Running simulation for mode: {mode_name}")
        mp.Simulation(cell_size=CELL, resolution=RESOLUTION, boundary_layers=[]).reset_meep()
        sources = [mp.EigenModeSource(src=mp.GaussianSource(FCEN, fwidth=DF),
                                      center=src_center, size=src_size, direction=mp.X,
                                      eig_band=props['band_num'], eig_parity=props['parity'])]
    
        sim = mp.Simulation(cell_size=CELL, boundary_layers=[mp.PML(DPML)],
                        geometry=full_geometry, sources=sources,
                        resolution=RESOLUTION, default_material=SIO2_MEDIUM)
    
        norm_flux = sim.add_mode_monitor(FCEN, DF, NFREQ, mp.ModeRegion(center=mp.Vector3(src_center.x + 0.2, 0), size=src_size))
        flux_p1 = sim.add_mode_monitor(FCEN, DF, NFREQ, mp.ModeRegion(center=mp.Vector3(monitor_x_pos, output_port1_y), size=monitor_size), eig_band=1)
        flux_p2 = sim.add_mode_monitor(FCEN, DF, NFREQ, mp.ModeRegion(center=mp.Vector3(monitor_x_pos, output_port2_y), size=monitor_size), eig_band=1)
        dft_monitor = sim.add_dft_fields([mp.Ez], FCEN, FCEN, 1, center=mp.Vector3(), size=mp.Vector3(SX, SY))
        
        sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ez, mp.Vector3(monitor_x_pos, 0), 1e-5))
        
        # Plotting & Analysis
        ez_dft_data = sim.get_dft_array(dft_monitor, mp.Ez, 0)
        eps_data = sim.get_epsilon()
        intensity = np.abs(ez_dft_data)**2
        
        # Calculate Transmission Power
        res_input = sim.get_eigenmode_coefficients(norm_flux, [props['band_num']], eig_parity=props['parity']).alpha[0, 0, 0]
        input_power = np.abs(res_input)**2 + 1e-12
        
        t1 = np.abs(sim.get_eigenmode_coefficients(flux_p1, [1]).alpha[0, 0, 0])**2 / input_power
        t2 = np.abs(sim.get_eigenmode_coefficients(flux_p2, [1]).alpha[0, 0, 0])**2 / input_power
        t1_db = 10 * np.log10(t1 + 1e-9)
        t2_db = 10 * np.log10(t2 + 1e-9)
    
        print(f"  [Result] {mode_name} -> Port1: {t1:.4f} ({t1_db:.2f} dB)")
        print(f"  [Result] {mode_name} -> Port2: {t2:.4f} ({t2_db:.2f} dB)")

        x = np.linspace(-SX/2, SX/2, intensity.shape[0])
        y = np.linspace(-SY/2, SY/2, intensity.shape[1])
    
        fig, ax = plt.subplots(figsize=(10, 6))
        # 這裡改成對應主程式的 colormap (inferno)
        im = ax.imshow(intensity.T, extent=[x.min(), x.max(), y.min(), y.max()], 
                   cmap='jet', origin='lower')
        ax.contour(eps_data.T, extent=[x.min(), x.max(), y.min(), y.max()], 
                   levels=[(N_SI**2+N_SIO2**2)/2], colors='white', alpha=0.5, linewidths=1, origin='lower')
        
        from mpl_toolkits.axes_grid1 import make_axes_locatable
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im, cax=cax, label='Intensity $|E_z|^2$')
        
        ax.set_title(f"DEMUX Final Structure ({mode_name})", fontsize=16)
        ax.set_xlabel("x ($\\mu$m)", fontsize=14)
        ax.set_ylabel("y ($\\mu$m)", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=14)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"Optimized_Final_Structure_{mode_name}.png"), dpi=300)
        plt.close(fig)
        
        print(f"Saved plot for {mode_name} to {output_folder}")

if __name__ == "__main__":
    # 填寫你的 npy 路徑。
    # 由於你的主程式 PARAMS 預設是 16x16，如果你輸出的是 best_config_16x16.npy，
    # 這邊可以直接對應填寫檔案路徑。
    npy_file = "/home/oscar3102/跑論文的圖/DEMUX-design3/best_config_10x10.npy"  # 請替換為您實際的 npy 檔案路徑
    output_dir = "/home/oscar3102/跑論文的圖/DEMUX-design3"
    
    # 這裡的預設參數也改為 16x16 對應你在主程式中的設定
    replot_from_npy(npy_file, output_dir, grid_rows=10, grid_cols=10)


Loading data from: /home/oscar3102/跑論文的圖/DEMUX-design3/best_config_10x10.npy
Auto-detected grid size: 10 x 10

>>> Starting Detailed MEEP Analysis & Plotting <<<
Running simulation for mode: TE0
-----------
Initializing structure...
time for choose_chunkdivision = 0.000453949 s
Working in 2D dimensions.
Computational cell is 7 x 6 x 0 with resolution 40
     block, center = (-2.75,0,0)
          size (1.5,1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (2.75,1,0)
          size (1.5,0.5,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (2.75,-1,0)
          size (1.5,0.5,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (0,0,0)
          size (4,4,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
time for s